This notebook is intended to do 5 final checks/fixes on the final timelines:
- Split entities that contain '/' (other than a/c) or 'and'
- Deduplicate identical tuples
- Remove tuples providing no additional info based on relation type (contains-1 uninformative) 
- Remove tuples providing no additional info based on mentions (general chemo mention uninformative)
- Make sure all SACT mentions are actually present in gold entities list or Vijay's extracted entities list
    - Currently have option toggled to not do this (consider_entities=False) because at least for task 2, removing SACT mentions not on the list decreases performance
- ~~Normalize all general chemo mentions to the first mention in the timeline~~

In [1]:
# import necessary stuff
import pandas as pd
import numpy as np
import copy
import os
import json

In [2]:
def split_combined_entities(timeline):
    timeline_copy = copy.deepcopy(timeline)
    result = []
    for triple in timeline_copy:
        entity = triple[0]
        rel = triple[1]
        date = triple[2]
        split_entity = entity.split('and')
        if len(split_entity) > 1:
            for sub_entity in split_entity:
                result.append([sub_entity.strip(),rel,date])
        else:
            split_entity = entity.split('/')
            do_it = False
            if len(split_entity) > 1:
                # Have to be careful, don't want to do this w/ a/c and other combos
                do_it = True
                for ele in split_entity:
                    if len(ele) < 3:
                        do_it = False
                        break
            if do_it:
                for sub_entity in split_entity:
                    result.append([sub_entity,rel,date]) # no need to strip in this case
            else:
                # No combined entities -> just add normally
                result.append([entity,rel,date])
    return result                    
                

In [3]:
def deduplicate_identical_tuples(timeline):
    timeline_copy = copy.deepcopy(timeline)
    result = []
    while len(timeline_copy) > 0:
        pot_ele = timeline_copy.pop(0) # "first" element
        if pot_ele not in timeline_copy: # no duplicates
            result.append(pot_ele)
    return result

In [4]:
def remove_trivial_rels(timeline):
    # returns a new timeline w/ trivial rels removed
    result = []
    for triple in timeline:
        # If the relation is contains-1 (not as much info), make sure there isn't 
        # a corresponding triple with same SACT and date and more specific relation
        # (begins-on or ends-on).
        must_continue = False
        if triple[1] == 'contains-1':
            for other_triple in timeline:
                if (other_triple[0] == triple[0]) and (other_triple[1] in ['ends-on','begins-on']) and (other_triple[2] == triple[2]):
                    must_continue = True
                    break # out of inner for loop
        if must_continue:
            continue # doesn't add triple to result
        # If no reason to cancel, just add to result
        result.append(triple)
    return result

In [5]:
CHEMO_MENTIONS = {
    "chemotherapy",
    "chemo",
    "chem",
    "chemo therapy",
    "chemo-radiation",
    "chemo-rt",
    "chemoembolization",
    "chemorad",
    "chemoirradiation",
    "chemort",
    "chemotherapeutic",
    "chemotherap",
    "chemotherapies",
    "chemotherapeutic",
    "chemotherapy's",
    "chemotheray",
    "chemoradiation",
} # global set variable

def remove_trivial_mentions(timeline):
    result = []
    for triple in timeline:
        # If the SACT is  a general mention (not as much info), make sure there isn't 
        # a corresponding triple with same relation and date and more specific SACT
        # (anything not a general mention.
        must_continue = False
        if triple[0] in CHEMO_MENTIONS:
            for other_triple in timeline:
                if (other_triple[0] not in CHEMO_MENTIONS) and (other_triple[1] == triple[1]) and (other_triple[2] == triple[2]):
                    must_continue = True
                    break # out of inner for loop
        if must_continue:
            continue # doesn't add triple to result
        # If no reason to cancel, just add to result
        result.append(triple)
    return result

In [6]:
# Import in synonyms and make function to return synonyms dict for given
# list of SACTs
synonym_file = "/data/project/alstate/chemoTimeline2025/test_results/2025-03-27 10-53-33.concept_synonym_stage.csv"
synonym_df = pd.read_csv(synonym_file)
synonym_df['synonym_name_lower'] = synonym_df['synonym_name'].apply(lambda x: x.lower())
print(synonym_df.head())

def get_synonyms_dict(gold_sacts):
    # the keys will be the lowercase synonyms,and the values the corresponding gold sact
    result = dict()
    for sact in gold_sacts:
        # Check if it is a general chemo mention. 
        if sact in CHEMO_MENTIONS:
            for mention in CHEMO_MENTIONS:
                result[mention] = sact
        else:
            # Otherwise, check the synonym df
            pot_codes = list((synonym_df[synonym_df['synonym_name_lower'] == sact])['synonym_concept_code'])
            if len(pot_codes) > 0: # don't care if no synonyms
                code = pot_codes[0]
                all_syns = list((synonym_df[synonym_df['synonym_concept_code'] == code])['synonym_name_lower'])
                for syn in all_syns:
                    result[syn] = sact
    return result

   synonym_concept_id synonym_name  synonym_concept_code  \
0                 NaN      AMG 330                     1   
1                 NaN      AMG-330                     1   
2                 NaN       AMG330                     1   
3                 NaN  Eluvixtamab                     1   
4                 NaN      AZD5363                     2   

  synonym_vocabulary_id  language_concept_id valid_start_date valid_end_date  \
0                HemOnc              4180186       2019-05-27     2025-03-23   
1                HemOnc              4180186       2023-08-19     2025-03-23   
2                HemOnc              4180186       2023-08-19     2025-03-23   
3                HemOnc              4180186       2019-05-27     2025-03-23   
4                HemOnc              4180186       2019-06-24     2025-03-23   

  invalid_reason synonym_name_lower  
0              U            amg 330  
1              U            amg-330  
2              U             amg330  
3     

In [7]:
def remove_ungold_mentions_and_rels(timeline,gold_sact,gold_rel):
    # returns a new timeline with tuples removed if they have a
    # mention or rel that doesn't appear in gold sacts or rels
    result = []
    syns_dict = get_synonyms_dict(gold_sact)
    for ele in timeline:
        if (ele[1] in gold_rel):
            if ele[0] in gold_sact:
                result.append(ele)
            else:
                # Change the sact if it is a synonym of a gold sact but not an actual gold sact
                if ele[0] in syns_dict:
                    new_ele = copy.deepcopy(ele)
                    new_ele[0] = syns_dict[ele[0]]
                    result.append(new_ele)
        # Note: if ele[1] not in gold_rel, don't even look for synonyms 
    return result

def remove_ungold_rels(timeline,gold_rel):
    # In case we don't have gold_sact
    result = []
    syns_dict = get_synonyms_dict(gold_sact)
    for ele in timeline:
        if (ele[1] in gold_rel):
            result.append(ele) 
    return result

In [8]:
def normalize_general_mentions(timeline):
    result = []
    first_chemo_mention = None
    for ele in timeline:
        ele_copy = copy.deepcopy(ele)
        if ele_copy[0] in CHEMO_MENTIONS:
            if first_chemo_mention != None:
                ele_copy[0] = first_chemo_mention
            else:
                first_chemo_mention = ele_copy[0]
        result.append(ele_copy)
    return result

In [9]:
def get_all_gold_entities(base_dir,cancer,patient):
    # cancer should be in ["ovarian","breast","melanoma"]
    # patient should be a string: patient##
    
    # Bug: currently this gets all doc times and events; we want events only
    # Fix: look at the next line and if it is <type>EVENT</event> it's fine
    specific_dir = base_dir + cancer + "/"
    specific_dir = specific_dir + cancer + "_" + patient + "_test/"
    # There should be a list of folders, one for each note
    note_folders = os.listdir(specific_dir)
    all_entities = []
    for note_folder in note_folders:
        note_folder_path = os.path.join(specific_dir,note_folder)
        #print(note_folder_path)
        entities_file_name = note_folder_path + "/" + note_folder + ".Temporal_Relation.gold.entity_only.xml"
        note_name = note_folder_path + "/" + note_folder
        #print("My file name:",entities_file_name)
        #print("Files in there:",os.listdir(note_folder_path))
        # Open the xml file
        # Turns out we can do this w/o xml parsing
        #tree = ET.parse(entities_file_name)
        xml_lines = []
        # It should bug around here if the file doesn't exist due to being
        # no gold entities
        with open(entities_file_name,"r") as file:
            xml_lines = file.readlines()
        # Open the note and read its contents
        note_contents = ""
        with open(note_name,"r") as file:
            note_contents = file.read()
        #root = tree.getroot()
        #anno = root.find('annotations') # the first and only one
        #for entity_and_more in anno.findall('entity'):
            #span_node = entity_and_more.find('span') # had issues getting attributes
        for line in xml_lines:
            # only care about lines with <span>. This should have start,end framed
            # by <span> on left and </span> on right
            stripped_line = line.strip()
            if "<span>" in stripped_line:
                span_str = stripped_line.split("<span>")[-1].split("</span>")[0]
                split_span = span_str.split(",")
                #print("split span:",split_span)
                start_i = int(split_span[0])
                end_i = int(split_span[1])
                entity = note_contents[start_i:end_i].lower()
                #print("Current entity:",entity)
                all_entities.append(entity)
            elif "<type>" in stripped_line:
                # If it is not of type EVENT, throw out the last entity (it is DOCTIME)
                type_str = stripped_line.split("<type>")[-1].split("</type>")[0]
                if type_str != "EVENT":
                    all_entities.pop()
    return list(set(all_entities))
            

In [10]:
def get_all_predicted_rels_all_patients(rels_file):
    # Note: rels_file should be the exact location of the json file,
    # not relative
    # Note: this doesn't do for all cancer types, just a single one
    # We can just evaluate the json and Python will turn it into a dict :)
    patient_rels_dict = dict()
    with open(rels_file,"r") as f:
        patient_rels_dict = eval(f.read())
    # We want to return a dict with just lists of rels for each patient
    result = dict()
    for patient in patient_rels_dict:
        # Values are timeline-like, but not normalized.
        # rels are the 2nd element of each triple
        timeline_like = patient_rels_dict[patient]
        curr_rels = []
        for triplet in timeline_like:
            curr_rels.append(triplet[1]) # python starts counting at 0
        curr_rels = list(set(curr_rels)) # deduplicate
        result[patient] = curr_rels
    return result
        

In [11]:
def get_all_predicted_entities_all_patients_all_cancers(entities_file):
    # Input is a csv.
    # Output is a dictionary of dictionaries:
    # {"breast":{"patient001":[SACTs]...}...}
    entities_df = pd.read_csv(entities_file) # note_path, events_list, timex_list
    result = {"breast":dict(),"ovarian":dict(),"melanoma":dict()}
    for row in entities_df.itertuples():
        curr_path = row.note_path
        curr_path_list = curr_path.split('/')
        # 4th from the end is the current cancer
        curr_cancer = curr_path_list[-4]
        # 2nd from the end is the current patient
        curr_patient = curr_path_list[-2]
        # Go through the events_list and add to growing list, deduplicated at the end
        pre_entities_list = eval(row.events_list) # now a list of dicts
        #if (curr_cancer == "ovarian" and curr_patient == "patient50"):
            #print("pre_entities_list:",pre_entities_list)
        entities_list = []
        for ele in pre_entities_list:
            curr_entity = ele['entity']
            entities_list.append(curr_entity)
        entities_list = list(set(entities_list))
        # Each (cancer, patient###) isn't be unique, as there can be multiple notes
        # for the same patient. Try appending it to the end of the dictionary value.
        # If this doesn't work, it hasn't been initialized -> initialize it. 
        try:
            (result[curr_cancer])[curr_patient].extend(entities_list)
        except:
            (result[curr_cancer])[curr_patient] = entities_list
    # At the end, still need to do some deduplication due to multiple notes
    for cancer in result:
        for patient in result[cancer]:
            result[cancer][patient] = list(set(result[cancer][patient]))
    return result

In [12]:
def task1(timelines_file_dict,gold_entities_dir,tlinks_file_dict,final_timelines_dir,consider_entities=True):
    """timelines_file_dict and tlinks_file_dict should both be of
    the form: {"breast":<breast file>, "ovarian":<ovarian file>, "melanoma":<melanoma file>}.
    All file names should be exact, not relative.
    """
    timeline_count = 0
    fixed_timeline_count = 0
    for cancer in ["breast","ovarian","melanoma"]:
        # Obtain the dictionary of timelines
        timelines_dict = dict()
        with open(timelines_file_dict[cancer],"r") as f:
            timelines_dict = eval(f.read()) # it is a json, should evaluate to dict
        # Obtain tlinks - gold entities will be obtained on the fly
        # Unless no info, then modifiy this or just don't use the dict
        #tlinks_dict = get_all_predicted_rels_all_patients(tlinks_file_dict[cancer])
        result_dict = dict() # same format as the json
        for patient in timelines_dict:
            entities_and_rels_working = consider_entities
            curr_timeline = timelines_dict[patient]
            # ran into issue if there any no entities or no relations. 
            # tries to look up entities and can't find them. 
            # In this case, the timeline is [], so don't attempt to fix it
            # and add it directly to the timeline.
            if curr_timeline == []:
                result_dict[patient] = curr_timeline
                timeline_count += 1
                continue # skip the rest of the stuff this iteration
            curr_entities = None
            curr_tlinks = ["contains-1","begins-on","ends-on"] # previously None
            try:
                # If error occurring here, likely due to not finding a gold entity list.
                # In this case, should return [] because no gold entities. 
                curr_entities = get_all_gold_entities(gold_entities_dir,cancer,patient)
                #curr_tlinks = tlinks_dict[patient]
            except FileNotFoundError as e:
                # Need to test to make sure this works
                print("An excpetion occurred:\n",e)
                result_dict[patient] = []
                timeline_count += 1
                fixed_timeline_count += 1
                continue # should go to next place in outer for loop
            fixed_timeline = split_combined_entities(curr_timeline)
            fixed_timeline = deduplicate_identical_tuples(fixed_timeline)
            fixed_timeline = remove_trivial_rels(fixed_timeline)
            fixed_timeline = remove_trivial_mentions(fixed_timeline)
            # if issue, then don't mess with these mentions and rels
            if entities_and_rels_working:
                # use ['contains-1','begins-on','ends-on'] if no tlinks info
                fixed_timeline = remove_ungold_mentions_and_rels(fixed_timeline,curr_entities,curr_tlinks)
            #fixed_timeline = normalize_general_mentions(fixed_timeline) # not doing anymore
            if fixed_timeline != curr_timeline:
                print("Had to change timeline")
                print("Original timeline:",curr_timeline)
                print("Fixed timeline:",fixed_timeline)
                fixed_timeline_count += 1
            result_dict[patient] = fixed_timeline
            timeline_count += 1
        # Now results_dict should contain all we need. Need to output it as a json.
        split_input_name = timelines_file_dict[cancer].split("/")
        original_short_name = split_input_name[-1]
        final_short_name = original_short_name.split(".")[0] + "_final.json"
        final_long_name = final_timelines_dir + final_short_name
        result_str = json.dumps(result_dict)
        with open(final_long_name,"w") as f:
            f.write(result_str)
    print("Proportion of timelines that needed fixing:",fixed_timeline_count/timeline_count)

In [13]:
def task2(timelines_file_dict,predicted_entities_file,tlinks_file_dict,final_timelines_dir,consider_entities=True):
    """timelines_file_dict and tlinks_file_dict should both be of
    the form: {"breast":<breast file>, "ovarian":<ovarian file>, "melanoma":<melanoma file>}.
    All file names should be exact, not relative.
    """
    # The predicted entities are all in one file
    entities_super_dict = get_all_predicted_entities_all_patients_all_cancers(predicted_entities_file)
    timeline_count = 0
    fixed_timeline_count = 0
    
    for cancer in ["breast","ovarian","melanoma"]:
        # Obtain the dictionary of timelines
        timelines_dict = dict()
        with open(timelines_file_dict[cancer],"r") as f:
            timelines_dict = eval(f.read()) # it is a json, should evaluate to dict
        # Obtain tlinks - gold entities will be obtained on the fly
        # If we don't have tlinks just use ['contains-1','begins-on','ends-on']
        #tlinks_dict = get_all_predicted_rels_all_patients(tlinks_file_dict[cancer])
        result_dict = dict() # same format as the json
        for patient in timelines_dict:
            curr_timeline = timelines_dict[patient]
            curr_entities = entities_super_dict[cancer][patient]
            #curr_tlinks = tlinks_dict[patient]
            curr_tlinks = ['contains-1','begins-on','ends-on']
            fixed_timeline = split_combined_entities(curr_timeline)
            fixed_timeline = deduplicate_identical_tuples(fixed_timeline)
            fixed_timeline = remove_trivial_rels(fixed_timeline) 
            fixed_timeline = remove_trivial_mentions(fixed_timeline)
            # Use ['contains-1','begins-on','ends-on'] if no tlinks info
            if consider_entities:
                fixed_timeline = remove_ungold_mentions_and_rels(fixed_timeline,curr_entities,curr_tlinks)
            #fixed_timeline = normalize_general_mentions(fixed_timeline) # not doing anymore
            if fixed_timeline != curr_timeline:
                print("Had to change timeline")
                print("Original timeline:",curr_timeline)
                print("Fixed timeline:",fixed_timeline)
                fixed_timeline_count += 1
            result_dict[patient] = fixed_timeline
            timeline_count += 1
        # Now results_dict should contain all we need. Need to output it as a json.
        split_input_name = timelines_file_dict[cancer].split("/")
        original_short_name = split_input_name[-1]
        final_short_name = original_short_name.split(".")[0] + "_final.json"
        final_long_name = final_timelines_dir + final_short_name
        result_str = json.dumps(result_dict)
        with open(final_long_name,"w") as f:
            f.write(result_str)
    print("Proportion of timelines that needed fixing:",fixed_timeline_count/timeline_count)

In [14]:
def task2_dev_test(timelines_file_dict,final_timelines_dir):
    """ Working with Chris's results on the dev data as a testing measure
    """
    timeline_count = 0
    fixed_timeline_count = 0
    for cancer in ["breast","ovarian","melanoma"]:
        # Obtain the dictionary of timelines
        timelines_dict = dict()
        with open(timelines_file_dict[cancer],"r") as f:
            timelines_dict = eval(f.read()) # it is a json, should evaluate to dict
        result_dict = dict() # same format as the json
        for patient in timelines_dict:
            curr_timeline = timelines_dict[patient]
            curr_tlinks = ["begins-on","ends-on","contains-1"] # most general case
            fixed_timeline = deduplicate_identical_tuples(curr_timeline)
            fixed_timeline = remove_trivial_rels(fixed_timeline)
            fixed_timeline = remove_trivial_mentions(fixed_timeline)
            fixed_timeline = remove_ungold_rels(fixed_timeline,curr_tlinks)
            fixed_timeline = normalize_general_mentions(fixed_timeline)
            if fixed_timeline != curr_timeline:
                print("Had to change timeline")
                print("Original timeline:",curr_timeline)
                print("Fixed timeline:",fixed_timeline)
                fixed_timeline_count += 1
            result_dict[patient] = fixed_timeline
            timeline_count += 1
        # Now results_dict should contain all we need. Need to output it as a json.
        split_input_name = timelines_file_dict[cancer].split("/")
        original_short_name = split_input_name[-1]
        final_short_name = original_short_name.split(".")[0] + "_final.json"
        final_long_name = final_timelines_dir + final_short_name
        result_str = json.dumps(result_dict)
        with open(final_long_name,"w") as f:
            f.write(result_str)
    print("Proportion of timelines that needed fixing:",fixed_timeline_count/timeline_count)

In [15]:
# Actually running stuff
# CHRIS EDIT HERE: you should only need to change the following two variables:
# - task1_timeline_input_file_dict
# - task2_timeline_input_file_dict

# Task 1
#task1_timeline_input_file_dict = {
    #"breast":"/data/project/alstate/chemoTimeline2025/test_results/final_timelines/subtask1_chris2/breast_test_all_patients_generated_timelines.json",
    #"ovarian":"/data/project/alstate/chemoTimeline2025/test_results/final_timelines/subtask1_chris2/ovarian_test_all_patients_generated_timelines.json",
    #"melanoma":"/data/project/alstate/chemoTimeline2025/test_results/final_timelines/subtask1_chris2/melanoma_test_all_patients_generated_timelines.json"
#}
task1_timeline_input_file_dict = {
    "breast":"/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3/breast_dev_all_patients_generated_timelines.json",
    "ovarian":"/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3/ovarian_dev_all_patients_generated_timelines.json",
    "melanoma":"/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3/melanoma_dev_all_patients_generated_timelines.json"
}
# The gold entities, as well as the tlinks, aren't actually used -> don't really matter
task1_gold_entities_dir = "/data/project/alstate/chemoTimeline2025/chemoTimelines2025_test_data/subtask1/Gold_PairWise_EventTimex/"
task1_tlinks_file_dict = {
    "breast":"/data/project/alstate/chemoTimeline2025/test_results/tlinks/breast_task1.json",
    "ovarian":"/data/project/alstate/chemoTimeline2025/test_results/tlinks/ovarian_task1.json",
    "melanoma":"/data/project/alstate/chemoTimeline2025/test_results/tlinks/melanoma_task1.json"
}
# Note: actually need to create the directory before running the program
task1_final_timeline_dir = "/data/project/alstate/chemoTimeline2025/test_results/final_timelines/subtask1_chris2_final/"

#task1(task1_timeline_input_file_dict,task1_gold_entities_dir,task1_tlinks_file_dict,task1_final_timeline_dir)

# Task 2
task2_timeline_input_file_dict = {
    "breast":"/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3/breast_dev_all_patients_generated_timelines.json",
    "ovarian":"/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3/ovarian_dev_all_patients_generated_timelines.json",
    "melanoma":"/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3/melanoma_dev_all_patients_generated_timelines.json"
}
# Entities: needs to be the right file. tlinks don't matter
task2_predicted_entities_file = "/data/project/alstate/chemoTimeline2025/test_results/dev/dev_entities_events_subtask2.csv"
task2_tlinks_file_dict = {
    "breast":"/data/project/alstate/chemoTimeline2025/test_results/tlinks/breast_task2.json",
    "ovarian":"/data/project/alstate/chemoTimeline2025/test_results/tlinks/ovarian_task2.json",
    "melanoma":"/data/project/alstate/chemoTimeline2025/test_results/tlinks/melanoma_task2.json"
}
#task2_final_timeline_dir = "/data/project/alstate/chemoTimeline2025/test_results/final_timelines/"
task2_final_timeline_dir = "/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3_final_8_19_ablate_entity/"

task2(task2_timeline_input_file_dict,task2_predicted_entities_file,task2_tlinks_file_dict,task2_final_timeline_dir,consider_entities=False)

Had to change timeline
Original timeline: [['cyclophosphamide', 'ends-on', '2012-02-29'], ['doxorubicin', 'ends-on', '2012-02-29']]
Fixed timeline: [['cytoxan', 'ends-on', '2012-02-29']]
Had to change timeline
Original timeline: [['adriamycin', 'contains-1', '2011'], ['adriamycin', 'contains-1', '2011-04-11'], ['cyclophosphamide', 'contains-1', '2011'], ['cytoxan', 'contains-1', '2011-04-11'], ['taxol', 'contains-1', '2011-04-11'], ['taxol', 'ends-on', '2011-12']]
Fixed timeline: [['adriamycin', 'contains-1', '2011'], ['adriamycin', 'contains-1', '2011-04-11'], ['cytoxan', 'contains-1', '2011'], ['cytoxan', 'contains-1', '2011-04-11'], ['taxol', 'contains-1', '2011-04-11'], ['taxol', 'ends-on', '2011-12']]
Had to change timeline
Original timeline: [['carboplatin', 'begins-on', '2012-01-12'], ['chemotherapy', 'begins-on', '2012-01-12'], ['chemotherapy', 'contains-1', '2012-01-12'], ['paclitaxel', 'begins-on', '2012-01-12']]
Fixed timeline: [['carboplatin', 'begins-on', '2012-01-12']]
Ha

In [16]:
# Testing
a = [1,1,2,3,4,5,0,6,-1,7]
b = deduplicate_identical_tuples(a)
print(a)
print(b)

timeline0 = [['a/c','contains-1','2005'],['carbo/taxol','begins-on','2005'],['carbotaxol','contains-1','2006'],['caboplatin and taxol','contains-1',2008]]
timeline0_a = split_combined_entities(timeline0)
print("timeline0:",timeline0)
print("timeline0_a:",timeline0_a)

timeline1 = [['ac','contains-1','2005'],['ac','begins-on','2005'],['ac','contains-1','2006']]
timeline1_a = remove_trivial_rels(timeline1)
print("timeline1:",timeline1)
print("timeline1_a:",timeline1_a)
timeline2 = [['ac','contains-1','2005'],['chemo','contains-1','2005'],['chemo','contains-1','2006']]
timeline2_a = remove_trivial_mentions(timeline2)
print("timeline2:",timeline2)
print("timeline2_a:",timeline2_a)

gold_sact = ['ac','chemo','tac','taxol','amg 330']
gold_rel = ['contains-1','begins-on','ends-on']
timeline3 = [['yeet','contains-1','2005'],['chemo','contains-1','2005'],['chemo','contains','2006'],['amg330','contains-1','2006']]
timeline3_a = remove_ungold_mentions_and_rels(timeline3,gold_sact,gold_rel)
print("timeline3:",timeline3)
print("timeline3_a:",timeline3_a)

timeline4 = [['yeet','contains-1','2004'],['chemo','contains-1','2005'],['chemotherapy','contains','2006'],['amg330','contains-1','2007']]
timeline4_a = normalize_general_mentions(timeline4)
print("timeline4:",timeline4)
print("timeline4_a:",timeline4_a)


base_dir = "/data/project/alstate/chemoTimeline2025/chemoTimelines2025_test_data/subtask1/Gold_PairWise_EventTimex/"
print("\nEntity extraction testing\n")
get_all_gold_entities(base_dir,"breast","patient02")

# Getting rels testing
#test_json_txt = ""
#with open("/data/project/alstate/chemoTimeline2025/test_results/tlinks/breast_task1.json","r") as f:
    #test_json_txt = f.read()
#test_json_eval = eval(test_json_txt)
#print(test_json_eval)
#print(type(test_json_eval))
print(get_all_predicted_rels_all_patients("/data/project/alstate/chemoTimeline2025/test_results/tlinks/breast_task1.json"))
print(get_all_predicted_entities_all_patients_all_cancers("/data/project/alstate/chemoTimeline2025/test_results/events_mods_timex/events_timex.csv"))

print("\nTesting pipeline on dev task 2 run 3:\n")
dev_timeline_input_file_dict = {
    "breast":"/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3/breast_dev_all_patients_generated_timelines.json",
    "ovarian":"/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3/ovarian_dev_all_patients_generated_timelines.json",
    "melanoma":"/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_round3/melanoma_dev_all_patients_generated_timelines.json"
}
dev_final_timeline_dir = "/data/project/alstate/chemoTimeline2025/test_results/dev/generated_timelines_final_round3/"
#task2_dev_test(dev_timeline_input_file_dict,dev_final_timeline_dir)

[1, 1, 2, 3, 4, 5, 0, 6, -1, 7]
[1, 2, 3, 4, 5, 0, 6, -1, 7]
timeline0: [['a/c', 'contains-1', '2005'], ['carbo/taxol', 'begins-on', '2005'], ['carbotaxol', 'contains-1', '2006'], ['caboplatin and taxol', 'contains-1', 2008]]
timeline0_a: [['a/c', 'contains-1', '2005'], ['carbo', 'begins-on', '2005'], ['taxol', 'begins-on', '2005'], ['carbotaxol', 'contains-1', '2006'], ['caboplatin', 'contains-1', 2008], ['taxol', 'contains-1', 2008]]
timeline1: [['ac', 'contains-1', '2005'], ['ac', 'begins-on', '2005'], ['ac', 'contains-1', '2006']]
timeline1_a: [['ac', 'begins-on', '2005'], ['ac', 'contains-1', '2006']]
timeline2: [['ac', 'contains-1', '2005'], ['chemo', 'contains-1', '2005'], ['chemo', 'contains-1', '2006']]
timeline2_a: [['ac', 'contains-1', '2005'], ['chemo', 'contains-1', '2006']]
timeline3: [['yeet', 'contains-1', '2005'], ['chemo', 'contains-1', '2005'], ['chemo', 'contains', '2006'], ['amg330', 'contains-1', '2006']]
timeline3_a: [['chemo', 'contains-1', '2005'], ['amg 330', 